In [17]:
import csv
import json
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import yaml

In [18]:
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.popularity import PopularityConfig, PopularityModel
from flowrec.models.svd import SVDConfig, SVDModel


In [19]:
SEEDS = (42, 43, 44)

In [20]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [21]:
REPO_ROOT = Path("..").resolve()
CONFIG_DIR = REPO_ROOT / "configs" / "exp1-standard_params"

# Dataset registries + shared data params come from the experiment configs:
# - dataset_amazon.yaml: beauty (2014), instruments (2018)
# - dataset_amazon_2023.yaml: beauty_2023
DATASET_CONFIG_FILES = (
    CONFIG_DIR / "dataset_amazon.yaml",
    CONFIG_DIR / "dataset_amazon_2023.yaml",
)


def load_yaml(path: Path) -> dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


# name -> {"entry": per-dataset registry entry, "data": shared params of its config file}
AMAZON_DATASETS: dict[str, dict[str, Any]] = {}

for config_path in DATASET_CONFIG_FILES:
    data_cfg = load_yaml(config_path)
    for name, entry in data_cfg.get("datasets", {}).items():
        AMAZON_DATASETS[name] = {"entry": entry, "data": data_cfg}

print("Datasets:", ", ".join(AMAZON_DATASETS))

Datasets: beauty, instruments, beauty_2023


In [ ]:
def make_data_config(dataset_cfg: dict[str, Any], seed: int) -> AmazonReviewDataConfig:
    """Build AmazonReviewDataConfig from the YAML experiment config.

    Follows the same convention as scripts/tiger_amazon_datasets.py::build_datamodule:
    per-dataset entry values override the file-level shared params.
    """
    entry = dataset_cfg["entry"]
    data = dataset_cfg["data"]
    num_workers = int(data.get("num_workers", 0))

    return AmazonReviewDataConfig(
        reviews_path=REPO_ROOT / entry["reviews_path"],
        metadata_path=REPO_ROOT / entry["metadata_path"] if entry.get("metadata_path") else None,
        item_key=entry.get("item_key", "asin"),
        metadata_item_key=entry.get("metadata_item_key", "parent_asin"),
        filter_reviews_to_metadata=bool(data.get("filter_reviews_to_metadata", True)),

        # Positive/auxiliary partition: matching rows form the positive training
        # stream (and define the catalog); the complement keeps its payload
        # columns and is exposed as dm.interaction_store for reward generation
        # and negative sampling.
        positive_criterion=entry.get("positive_criterion", data.get("positive_criterion")),

        # Splitting.
        split_strategy=data.get("split_strategy", "leave_one_out"),
        min_history_len=int(data.get("min_history_len", 1)),
        train_all_prefixes=bool(data.get("train_all_prefixes", True)),

        # Filtering.
        min_user_interactions=int(data.get("min_user_interactions", 5)),
        min_item_interactions=int(data.get("min_item_interactions", 5)),
        max_history_len=int(data.get("max_history_len", 20)),
        deduplicate_consecutive=bool(data.get("deduplicate_consecutive", False)),
        filter_cold_items_after_split=bool(data.get("filter_cold_items_after_split", False)),
        cold_item_train_source=data.get("cold_item_train_source", "observed"),
        offset_semantic_ids_by_one=bool(data.get("offset_semantic_ids_by_one", True)),

        # Batching.
        batch_size=int(data.get("batch_size", 128)),
        eval_batch_size=int(data.get("eval_batch_size", 64)),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
        seed=seed,
    )

In [23]:
def fit_popularity_model(dm) -> PopularityModel:
    num_items = dm.item_vocab_size

    model = PopularityModel(
        PopularityConfig(
            num_items=num_items,
            pad_item_id=0,
            score_transform="log1p",
            exclude_history_in_generate=False,
        )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None

    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        try:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets_only=False,
            )
        except TypeError:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets=True,
                count_history=True,
            )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_targets=True,
            count_history=True,
        )

    return model


def fit_svd_model(dm) -> SVDModel:
    num_users = dm.num_users
    num_items = dm.item_vocab_size

    model = SVDModel(
        SVDConfig(
            num_users=num_users,
            num_items=num_items,
            n_components=128,
            input_transform="binary",
            user_representation="history_mean",
            foldin_reg=1e-2,
            exclude_history_in_generate=False,
            )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None

    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        model.fit_from_store(
            store=store,
            target_index=train_index,
            fit_source="observed_prefix",
            use_ratings=False,
        )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_history=True,
            count_targets=True,
            use_ratings=False,
        )

    return model

In [24]:
@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    *,
    num_items: int,
    ks: tuple[int, ...] = (5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    model.eval()

    max_k = max(ks)

    total = 0
    hit_sum = {k: 0.0 for k in ks}
    ndcg_sum = {k: 0.0 for k in ks}
    coverage_mask = {k: torch.zeros(num_items, dtype=torch.bool) for k in ks}

    for batch in data_loader:
        target = batch.get("target_item_ids", batch.get("target_item_id"))

        if target is None:
            raise KeyError("Batch must contain target_item_ids or target_item_id.")

        target = target.long()

        recs = model.generate(
            batch,
            num_return_sequences=max_k,
        ).cpu()

        valid = target != pad_item_id
        valid &= target >= 0
        valid &= target < num_items

        if valid.sum() == 0:
            continue

        target = target[valid].cpu()
        recs = recs[valid]

        batch_size = target.size(0)
        total += batch_size

        for k in ks:
            topk = recs[:, :k]

            matches = topk.eq(target.unsqueeze(1))
            hit = matches.any(dim=1)

            hit_sum[k] += float(hit.float().sum().item())

            rank = matches.float().argmax(dim=1)
            ndcg = torch.zeros(batch_size, dtype=torch.float32)

            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank[hit].float() + 2.0)

            ndcg_sum[k] += float(ndcg.sum().item())

            valid_recs = topk[
                (topk != pad_item_id)
                & (topk >= 0)
                & (topk < num_items)
            ]

            if valid_recs.numel() > 0:
                coverage_mask[k][valid_recs.unique()] = True

    if total == 0:
        raise RuntimeError("No valid examples found.")

    catalog_size = max(num_items - 1, 1)

    metrics: dict[str, float] = {}

    for k in ks:
        metrics[f"hit_rate@{k}"] = hit_sum[k] / total
        metrics[f"ndcg@{k}"] = ndcg_sum[k] / total
        metrics[f"coverage@{k}"] = float(coverage_mask[k][1:].float().mean().item())

    metrics["num_examples"] = float(total)
    metrics["num_items"] = float(catalog_size)

    return metrics


In [25]:
def run_one_model(
    *,
    model_name: str,
    dm,
) -> dict[str, float]:
    if model_name == "most_popular":
        model = fit_popularity_model(dm)
    elif model_name == "svd":
        model = fit_svd_model(dm)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    metrics: dict[str, float] = {}

    for split_name, data_loader in (
        ("val", dm.val_dataloader()),
        ("test", dm.test_dataloader()),
    ):
        split_metrics = evaluate_model(
            model,
            data_loader,
            num_items=dm.item_vocab_size,
            ks=(5, 10, 20),
            pad_item_id=0,
        )

        print(f"{split_name.capitalize()} metrics:")
        for key in sorted(split_metrics):
            print(f"  {key}: {split_metrics[key]:.6f}")

        metrics.update({f"{split_name}/{key}": value for key, value in split_metrics.items()})

    return metrics


def run_one_seed(
    dataset_name: str,
    dataset_cfg: dict[str, Any],
    seed: int,
) -> list[dict[str, Any]]:
    print(f"\n=== Dataset={dataset_name} | seed={seed} ===")

    set_seed(seed)

    data_cfg = make_data_config(dataset_cfg, seed)

    dm = AmazonReviewDataModule(data_cfg)
    dm.setup()

    # Includes positive_criterion and the interaction_store breakdown
    # (auxiliary events kept for rewards / negative sampling).
    print(json.dumps(dm.summary(), indent=2, default=str))

    rows: list[dict[str, Any]] = []

    for model_name in ("most_popular", "svd"):
        print(f"Running model: {model_name}")

        try:
            metrics = run_one_model(
                model_name=model_name,
                dm=dm,
            )

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                **metrics,
            }

        except MemoryError as exc:
            print(f"Skipping {model_name} due to MemoryError: {exc}")

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                "error": f"MemoryError: {exc}",
            }

        rows.append(row)

    return rows


def aggregate_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    base_metric_keys = [
        "hit_rate@5",
        "hit_rate@10",
        "hit_rate@20",
        "ndcg@5",
        "ndcg@10",
        "ndcg@20",
        "coverage@5",
        "coverage@10",
        "coverage@20",
        "num_examples",
        "num_items",
    ]
    metric_keys = [
        f"{split_name}/{metric}"
        for split_name in ("val", "test")
        for metric in base_metric_keys
    ]

    groups: dict[tuple[str, str], list[dict[str, Any]]] = {}

    for row in rows:
        if "error" in row:
            continue
        key = (row["dataset"], row["model"])
        groups.setdefault(key, []).append(row)

    summary_rows: list[dict[str, Any]] = []

    for (dataset, model), group_rows in groups.items():
        summary: dict[str, Any] = {
            "dataset": dataset,
            "model": model,
            "seed": "mean",
        }

        for metric in metric_keys:
            values = [
                float(row[metric])
                for row in group_rows
                if metric in row
            ]

            if not values:
                continue

            values_np = np.asarray(values, dtype=np.float64)
            summary[metric] = float(values_np.mean())
            summary[f"{metric}_std"] = float(values_np.std(ddof=0))

        summary_rows.append(summary)

    return summary_rows


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: list[str] = []

    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
output_dir = Path("../artifacts/metrics/amazon_baselines")
output_dir.mkdir(parents=True, exist_ok=True)

per_seed_rows: list[dict[str, Any]] = []

for dataset_name, dataset_cfg in AMAZON_DATASETS.items():
    for seed in SEEDS:
        per_seed_rows.extend(
            run_one_seed(
                dataset_name=dataset_name,
                dataset_cfg=dataset_cfg,
                seed=seed,
            )
        )

summary_rows = aggregate_rows(per_seed_rows)

write_csv(output_dir / "per_seed_metrics.csv", per_seed_rows)
write_csv(output_dir / "summary_metrics.csv", summary_rows)

print(f"\nSaved per-seed metrics to: {output_dir / 'per_seed_metrics.csv'}")
print(f"Saved summary metrics to: {output_dir / 'summary_metrics.csv'}")



=== Dataset=beauty | seed=42 ===
{
  "num_users": 22363,
  "num_items": 12101,
  "num_train_examples": 131413,
  "num_val_examples": 22363,
  "num_test_examples": 22363,
  "num_interactions": 198502,
  "mean_history_len": 6.695047661222103,
  "mean_train_history_len": 6.701551596873977,
  "mean_val_history_len": 6.19872110181997,
  "mean_test_history_len": 7.153154764566471,
  "post_filter_cold_items": false,
  "cold_item_reference": "observed",
  "num_items_before_cold_filter": 12101,
  "num_cold_items_removed": 0,
  "num_train_examples_dropped_cold": 0,
  "num_val_examples_dropped_cold": 0,
  "num_test_examples_dropped_cold": 0,
  "num_train_history_items_removed_cold": 0,
  "num_val_history_items_removed_cold": 0,
  "num_test_history_items_removed_cold": 0,
  "split_strategy": "leave_one_out",
  "train_example_layout": "prefix_target",
  "global_val_cutoff": null,
  "global_test_cutoff": null,
  "sid_len": null,
  "item_vocab_size": 12102,
  "positive_criterion": null,
  "interacti

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/synth/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/synth/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
